# Experiment 3 — Same-Checkpoint Horizon/Offset Reliance


This experiment fixes **one trained iTransformer checkpoint with `pred_len=720`** and measures Expected Permutation Importance separately at offsets \(h\in\{96,192,336,720\}\).

Everything except the evaluated output offset is fixed: checkpoint, parameters, input windows, target/source pools, perturbation seeds, and test windows.

Outputs:
- per target–source–offset EPI,
- pairwise cross-offset Spearman/Jaccard stability,
- optional alignment between offset-specific controlled utility and same-checkpoint EPI.

**Interpretation is informative in either direction:** strong drift means `used` itself changes across future offsets inside one forecaster; strong stability means the trained forecaster uses a comparatively stable source strategy even though `related`/`useful` change.


In [1]:
from pathlib import Path
from types import SimpleNamespace
import gc, importlib, math, random, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

warnings.filterwarnings("ignore")
SEED = 2026
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEQ_LEN = 96
LABEL_LEN = 48
MODEL_H = 720
OFFSETS = [96, 192, 336, 720]
DATASETS = ["Electricity", "Weather", "Solar", "ETTh1", "ETTm1"]
ALL_HORIZONS = [96, 192, 336, 720]

MAX_TOPK = 10
RAW_CORR_TOPM = 8
TEACHER_CONSENSUS_TOPM = 12
MAX_TARGETS_HIGH_DIM = 6
MAX_TARGETS_WEATHER = 10
MAX_TRAIN_ORIGINS = 3000
MAX_TEST_WINDOWS = 256
PERTURB_SEEDS = [2026, 2027]
SOURCE_CHUNK = 8
RANK_K = 5
NUM_WORKERS = 2

PROJECT_ROOT = Path("/data/code/2026_08")
BASELINE_ROOT = PROJECT_ROOT / "results_external_baselines_phase1_direct"
TEACHER_CACHE_ROOT = (PROJECT_ROOT / "results_itransformer_explicit_sparse_horizon_mask" / "predictive_teacher_hardmask_cache")
OUTPUT_DIR = PROJECT_ROOT / "results_same_checkpoint_offset_reliance_H720"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

def effective_topk(C):
    return int(max(1, min(MAX_TOPK, math.ceil((C - 1) / 2))))

def choose_targets(dataset_name, C):
    if C <= 10:
        return np.arange(C, dtype=np.int64)
    n = MAX_TARGETS_WEATHER if dataset_name == "Weather" else MAX_TARGETS_HIGH_DIM
    return np.unique(np.linspace(0, C - 1, min(n, C), dtype=np.int64))

def choose_batch_size(dataset_name, H):
    if dataset_name == "Electricity":
        return 8 if H >= 336 else 12
    if dataset_name == "Solar":
        return 16
    return 32

def evenly_subsample(n, max_n):
    if n <= max_n:
        return np.arange(n, dtype=np.int64)
    return np.unique(np.linspace(0, n - 1, max_n, dtype=np.int64))

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Output:", OUTPUT_DIR)

PyTorch: 2.4.1+cu121
Device: cuda
Output: /data/code/2026_08/results_same_checkpoint_offset_reliance_H720


## 1. Time-Series-Library and dataset protocol

In [2]:
# Locate Time-Series-Library
TSLIB_CANDIDATES = [
    Path("/data/Time-Series-Library"),
    Path("/data/Time-Series-Library_v2"),
    Path("/data/Time-Series-Library"),
]

TSL_ROOT = next(
    (p.resolve() for p in TSLIB_CANDIDATES if (p / "models" / "iTransformer.py").exists()),
    None,
)

if TSL_ROOT is None:
    raise FileNotFoundError("Time-Series-Library source not found.")

if str(TSL_ROOT) not in sys.path:
    sys.path.insert(0, str(TSL_ROOT))

iTransformer_module = importlib.import_module("models.iTransformer")
print("TSLib:", TSL_ROOT)

TSLib: /data/Time-Series-Library


In [3]:
DATASET_CANDIDATES = {
    "Electricity": [
        "/data/dataset/electricity/electricity.csv",
        "/data/dataset/ECL/electricity.csv",
        "/data/dataset/electricity.csv",
    ],
    "Weather": [
        "/data/dataset/weather/weather.csv",
        "/data/dataset/weather.csv",
    ],
    "Solar": [
        "/data/dataset/solar/solar_AL.txt",
        "/data/dataset/Solar/solar_AL.txt",
        "/data/dataset/solar_AL.txt",
    ],
    "ETTh1": [
        "/data/dataset/ETT-small/ETTh1.csv",
        "/data/dataset/ETTh1.csv",
    ],
    "ETTm1": [
        "/data/dataset/ETT-small/ETTm1.csv",
        "/data/dataset/ETTm1.csv",
    ],
}

DATASET_SPECS = {
    "Electricity": {"channels": 321, "freq": "h"},
    "Weather": {"channels": 21, "freq": "t"},
    "Solar": {"channels": 137, "freq": "h"},
    "ETTh1": {"channels": 7, "freq": "h"},
    "ETTm1": {"channels": 7, "freq": "t"},
}

def resolve_path(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p.resolve()
    return None

DATASET_FILES = {}
for d, candidates in DATASET_CANDIDATES.items():
    path = resolve_path(candidates)
    if path is None:
        raise FileNotFoundError(f"{d}: dataset file not found.")
    DATASET_FILES[d] = path
    print(f"{d:12s} -> {path}")

def load_numeric_series(dataset_name, path):
    path = Path(path)
    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        date_col = None
        for c in ["date", "datetime", "timestamp", "time"]:
            if c in df.columns:
                date_col = c
                break
        if date_col is None:
            first = df.columns[0]
            if not pd.api.types.is_numeric_dtype(df[first]):
                date_col = first
        if date_col is not None:
            df = df.drop(columns=[date_col])
        df = df.select_dtypes(include=[np.number])
        x = df.to_numpy(dtype=np.float32)
    else:
        try:
            x = np.loadtxt(path, delimiter=",", dtype=np.float32)
        except Exception:
            x = np.loadtxt(path, dtype=np.float32)
        if x.ndim == 1:
            x = x[:, None]
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError(f"{dataset_name}: invalid data shape/content {x.shape}")
    return x

def split_boundaries(dataset_name, T):
    if dataset_name == "ETTh1":
        tr = 12 * 30 * 24
        va = tr + 4 * 30 * 24
        te = va + 4 * 30 * 24
        return min(tr, T), min(va, T), min(te, T)
    if dataset_name == "ETTm1":
        unit = 30 * 24 * 4
        tr = 12 * unit
        va = tr + 4 * unit
        te = va + 4 * unit
        return min(tr, T), min(va, T), min(te, T)
    return int(0.70 * T), int(0.80 * T), T

def load_and_normalize(dataset_name):
    raw = load_numeric_series(dataset_name, DATASET_FILES[dataset_name])
    T, C = raw.shape
    train_end, val_end, test_end = split_boundaries(dataset_name, T)
    mean = raw[:train_end].mean(axis=0, keepdims=True)
    std = np.maximum(raw[:train_end].std(axis=0, keepdims=True), 1e-6)
    x = (raw - mean) / std
    return x.astype(np.float32), {
        "T": T, "C": C,
        "train_end": train_end,
        "val_end": val_end,
        "test_end": test_end,
    }

class ForecastDataset(Dataset):
    def __init__(self, data, pred_len, start_min, end_exclusive):
        self.data = torch.from_numpy(data).float()
        self.pred_len = int(pred_len)
        first_t = max(SEQ_LEN, LABEL_LEN, int(start_min))
        last_t = int(end_exclusive) - self.pred_len
        self.origins = np.arange(first_t, last_t + 1, dtype=np.int64)
        if len(self.origins) == 0:
            raise RuntimeError("No valid windows.")

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, idx):
        t = int(self.origins[idx])
        x = self.data[t-SEQ_LEN:t]
        y = self.data[t-LABEL_LEN:t+self.pred_len]
        return x, y

Electricity  -> /data/dataset/electricity/electricity.csv
Weather      -> /data/dataset/weather/weather.csv
Solar        -> /data/dataset/solar/solar_AL.txt
ETTh1        -> /data/dataset/ETT-small/ETTh1.csv
ETTm1        -> /data/dataset/ETT-small/ETTm1.csv


In [4]:
def build_itransformer_args(dataset_name, H):
    spec = DATASET_SPECS[dataset_name]
    return SimpleNamespace(
        task_name="long_term_forecast",
        model="iTransformer",
        seq_len=SEQ_LEN,
        label_len=LABEL_LEN,
        pred_len=H,
        enc_in=spec["channels"],
        dec_in=spec["channels"],
        c_out=spec["channels"],
        e_layers=3,
        d_layers=1,
        n_heads=8,
        d_model=512,
        d_ff=512,
        factor=3,
        dropout=0.1,
        embed="timeF",
        freq=spec["freq"],
        activation="gelu",
        output_attention=False,
        class_strategy="projection",
        use_norm=1,
        moving_avg=25,
        distil=True,
        top_k=5,
        num_kernels=6,
        seasonal_patterns="Monthly",
        inverse=False,
    )

def build_and_load_itransformer(dataset_name, H):
    model = iTransformer_module.Model(build_itransformer_args(dataset_name, H)).float().to(DEVICE)
    checkpoint = BASELINE_ROOT / "iTransformer" / dataset_name / f"H{H}" / "best_model.pt"
    if not checkpoint.exists():
        raise FileNotFoundError(checkpoint)
    state = torch.load(checkpoint, map_location="cpu")
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

## 2. Training-only candidate pools and offset-specific predictive utility

In [5]:
# Training-only dependency features and teacher utilities
def common_train_origins(train_end):
    last = int(train_end) - max(ALL_HORIZONS)
    origins = np.arange(SEQ_LEN, last + 1, dtype=np.int64)
    return origins[evenly_subsample(len(origins), MAX_TRAIN_ORIGINS)]

def summary_features(x, origins):
    last = x[origins - 1]
    mean3 = np.stack([x[t-3:t].mean(axis=0) for t in origins], axis=0)
    mean12 = np.stack([x[t-12:t].mean(axis=0) for t in origins], axis=0)
    delta12 = x[origins - 1] - x[origins - 12]
    return np.stack([last, mean3, mean12, delta12], axis=-1).astype(np.float32)

def absolute_current_correlation(features):
    cur = features[:, :, 0].astype(np.float64)
    cur = cur - cur.mean(axis=0, keepdims=True)
    denom = np.maximum(np.sqrt(np.sum(cur * cur, axis=0)), 1e-12)
    z = cur / denom[None, :]
    corr = np.abs(z.T @ z)
    np.fill_diagonal(corr, 0.0)
    return corr.astype(np.float32)

def teacher_path(dataset_name, H, K):
    return TEACHER_CACHE_ROOT / f"{dataset_name}_H{H}_endpoint_predictive_topk_K{K}.npz"

def load_teacher_topk(dataset_name, H, C, K):
    path = teacher_path(dataset_name, H, K)
    if not path.exists():
        raise FileNotFoundError(path)
    c = np.load(path)
    idx = c["source_idx"].astype(np.int64)
    gain = c["source_gain"].astype(np.float32)
    if idx.shape != (C, K):
        raise RuntimeError(f"Teacher shape mismatch: {idx.shape} vs {(C,K)}")
    return idx, gain

def teacher_consensus_sources(dataset_name, C, K, topm):
    rank = {H: load_teacher_topk(dataset_name, H, C, K) for H in ALL_HORIZONS}
    result = {}
    for i in range(C):
        score, freq, gsum = {}, {}, {}
        for H in ALL_HORIZONS:
            idx, gain = rank[H]
            for r, (j, g) in enumerate(zip(idx[i].tolist(), gain[i].tolist())):
                j = int(j)
                if j == i:
                    continue
                score[j] = score.get(j, 0.0) + float(K - r)
                freq[j] = freq.get(j, 0) + 1
                gsum[j] = gsum.get(j, 0.0) + float(g)
        cand = list(score)
        cand.sort(key=lambda j: (score[j], freq[j], gsum[j]), reverse=True)
        result[i] = np.asarray(cand[:topm], dtype=np.int64)
    return result

def build_candidate_pools(dataset_name, corr, targets):
    C = corr.shape[0]
    if C <= 10:
        return {
            int(i): np.asarray([j for j in range(C) if j != i], dtype=np.int64)
            for i in targets
        }
    K = effective_topk(C)
    consensus = teacher_consensus_sources(dataset_name, C, K, TEACHER_CONSENSUS_TOPM)
    pools = {}
    for i0 in targets:
        i = int(i0)
        corr_src = [int(j) for j in np.argsort(corr[i])[::-1] if j != i][:RAW_CORR_TOPM]
        combined = []
        for j in corr_src + consensus[i].tolist():
            if j != i and j not in combined:
                combined.append(int(j))
        pools[i] = np.asarray(combined, dtype=np.int64)
    return pools

def add_bias(X):
    return np.concatenate([np.ones((len(X), 1)), X.astype(np.float64)], axis=1)

def ridge_fit(X, y, lam=1e-3):
    Xb = add_bias(X)
    reg = np.eye(Xb.shape[1], dtype=np.float64)
    reg[0, 0] = 0.0
    return np.linalg.solve(Xb.T @ Xb + lam * reg, Xb.T @ y.astype(np.float64))

def ridge_predict(X, beta):
    return add_bias(X) @ beta

def predictive_utility_for_sources(features, y, target_idx, source_ids):
    N = len(features)
    split = min(max(int(0.70 * N), 32), N - 16)
    fit = np.arange(split)
    score = np.arange(split, N)

    Xt = features[:, target_idx, :].astype(np.float64)
    U = features[:, source_ids, :].astype(np.float64)
    y = y.astype(np.float64)

    bt = ridge_fit(Xt[fit], y[fit], 1e-3)
    r_fit = y[fit] - ridge_predict(Xt[fit], bt)
    r_score = y[score] - ridge_predict(Xt[score], bt)
    base_mse = float(np.mean(r_score ** 2))

    Xf = add_bias(Xt[fit])
    Xs = add_bias(Xt[score])
    S, Fdim = U.shape[1], U.shape[2]

    Uf = U[fit].reshape(len(fit), S * Fdim)
    Us = U[score].reshape(len(score), S * Fdim)

    reg = np.eye(Xf.shape[1], dtype=np.float64)
    reg[0, 0] = 0.0
    coef = np.linalg.solve(Xf.T @ Xf + 1e-3 * reg, Xf.T @ Uf)

    Uf_res = (Uf - Xf @ coef).reshape(len(fit), S, Fdim)
    Us_res = (Us - Xs @ coef).reshape(len(score), S, Fdim)

    gram = np.einsum("nsf,nsg->sfg", Uf_res, Uf_res)
    cross = np.einsum("nsf,n->sf", Uf_res, r_fit)
    eye = np.eye(Fdim)[None, :, :]
    bs = np.linalg.solve(gram + 1e-2 * eye, cross[:, :, None])[:, :, 0]

    pred_res = np.einsum("nsf,sf->ns", Us_res, bs)
    mse = np.mean((r_score[:, None] - pred_res) ** 2, axis=0)

    return (100.0 * (base_mse - mse) / max(base_mse, 1e-12)).astype(np.float32)

In [6]:
# Test loader, latent hook, and perturbation
def prepare_sampled_test(dataset_name, H):
    x, meta = load_and_normalize(dataset_name)
    ds = ForecastDataset(x, H, meta["val_end"], meta["test_end"])
    idx = evenly_subsample(len(ds), MAX_TEST_WINDOWS)
    subset = Subset(ds, idx.tolist())
    loader = DataLoader(
        subset,
        batch_size=choose_batch_size(dataset_name, H),
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )
    return x, meta, loader, idx

class EncoderCapture:
    def __init__(self, encoder):
        self.value = None
        self.handle = encoder.register_forward_hook(self._hook)
    def _hook(self, module, inputs, output):
        self.value = (output[0] if isinstance(output, tuple) else output).detach()
    def close(self):
        self.handle.remove()

def nonzero_shift(B, seed, source_idx, batch_idx):
    if B <= 1:
        return None
    v = seed * 1000003 + int(source_idx) * 9176 + batch_idx * 131
    return 1 + (v % (B - 1))

def perturb_source_chunk(x, source_chunk, seed, batch_idx):
    B, L, C = x.shape
    S = len(source_chunk)
    aug = x[None].expand(S, B, L, C).clone()
    for q, source_idx in enumerate(source_chunk):
        j = int(source_idx)
        shift = nonzero_shift(B, seed, j, batch_idx)
        if shift is None:
            donor = torch.flip(x[:, :, j], dims=[1])
        else:
            donor_idx = torch.roll(torch.arange(B, device=x.device), shifts=shift, dims=0)
            donor = x[donor_idx, :, j]
        aug[q, :, :, j] = donor
    return aug.reshape(S * B, L, C)

## 3. Same-checkpoint per-offset EPI

In [7]:
@torch.inference_mode()
def run_same_checkpoint_dataset(dataset_name):
    H = MODEL_H
    set_seed(SEED)
    x_np, meta, loader, sample_idx = prepare_sampled_test(dataset_name, H)
    C = int(meta["C"])
    targets = choose_targets(dataset_name, C)

    train_origins = common_train_origins(meta["train_end"])
    train_feat = summary_features(x_np, train_origins)
    raw_corr = absolute_current_correlation(train_feat)
    pools = build_candidate_pools(dataset_name, raw_corr, targets)

    global_sources = np.asarray(sorted({int(j) for i in targets for j in pools[int(i)] if int(j) != int(i)}), dtype=np.int64)
    source_pos = {int(j): q for q, j in enumerate(global_sources)}
    offset_idx = {int(h): int(h - 1) for h in OFFSETS}

    print(f"{dataset_name} | fixed checkpoint H={H} | C={C} | targets={len(targets)} | sources={len(global_sources)} | test={len(sample_idx)}")

    # Offset-specific P on exactly the same candidate pools.
    P = {h: {} for h in OFFSETS}
    for i0 in targets:
        i = int(i0); src = pools[i]
        for h in OFFSETS:
            ytr = x_np[train_origins + h - 1, i]
            util = predictive_utility_for_sources(train_feat, ytr, i, src)
            P[h][i] = {int(j): float(u) for j, u in zip(src.tolist(), util.tolist())}

    model = build_and_load_itransformer(dataset_name, H)
    model.eval()
    Tn, Sn, On = len(targets), len(global_sources), len(OFFSETS)

    clean_sse = np.zeros((On, Tn), dtype=np.float64)
    clean_n = np.zeros((On, Tn), dtype=np.int64)
    pert_sse = {s: np.zeros((On, Tn, Sn), dtype=np.float64) for s in PERTURB_SEEDS}

    for bidx, (x, y) in enumerate(loader):
        x = x.float().to(DEVICE, non_blocking=True)
        y = y.float().to(DEVICE, non_blocking=True)
        truth = y[:, -H:, :]
        pred = model(x, None, None, None)[:, -H:, :]

        for oq, h in enumerate(OFFSETS):
            k = offset_idx[h]
            err = pred[:, k, targets] - truth[:, k, targets]
            clean_sse[oq] += err.square().sum(dim=0).cpu().numpy()
            clean_n[oq] += err.shape[0]

        for seed in PERTURB_SEEDS:
            for start in range(0, Sn, SOURCE_CHUNK):
                chunk = global_sources[start:start+SOURCE_CHUNK]
                x_aug = perturb_source_chunk(x, chunk, seed, bidx)
                p_aug = model(x_aug, None, None, None)[:, -H:, :]
                p_aug = p_aug.reshape(len(chunk), x.shape[0], H, C)
                for sq, j0 in enumerate(chunk):
                    sp = source_pos[int(j0)]
                    for oq, h in enumerate(OFFSETS):
                        k = offset_idx[h]
                        err = p_aug[sq, :, k, targets] - truth[:, k, targets]
                        pert_sse[seed][oq, :, sp] += err.square().sum(dim=0).cpu().numpy()
                del x_aug, p_aug
        if bidx % 10 == 0:
            print(f"  batch {bidx+1}/{len(loader)}")

    rows = []
    for tq, i0 in enumerate(targets):
        i = int(i0)
        for j0 in pools[i]:
            j = int(j0)
            if j == i: continue
            sp = source_pos[j]
            for oq, h in enumerate(OFFSETS):
                clean_mse = clean_sse[oq, tq] / max(clean_n[oq, tq], 1)
                imp = []
                for seed in PERTURB_SEEDS:
                    pmse = pert_sse[seed][oq, tq, sp] / max(clean_n[oq, tq], 1)
                    imp.append(100.0 * (pmse - clean_mse) / max(clean_mse, 1e-12))
                rows.append({
                    "dataset": dataset_name,
                    "model_pred_len": int(H),
                    "offset": int(h),
                    "target_channel": i,
                    "source_channel": j,
                    "raw_abs_corr": float(raw_corr[i, j]),
                    "predictive_utility_%": float(P[h][i][j]),
                    "same_checkpoint_epi_%": float(np.mean(imp)),
                    "epi_seed_std": float(np.std(imp)),
                    "candidate_count": int(len(pools[i])),
                    "test_windows": int(len(sample_idx)),
                })

    result = pd.DataFrame(rows)
    del model, loader, train_feat, raw_corr, pert_sse
    gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return result


## 4. Run all H=720 checkpoints with resume

In [8]:
frames, failures = [], []
for dataset_name in DATASETS:
    path = OUTPUT_DIR / f"{dataset_name}_H720_offset_epi.csv"
    if path.exists():
        print("Reusing", path)
        frames.append(pd.read_csv(path))
        continue
    try:
        df = run_same_checkpoint_dataset(dataset_name)
        df.to_csv(path, index=False)
        frames.append(df)
    except Exception as e:
        print("FAILED", dataset_name, repr(e))
        failures.append({"dataset": dataset_name, "error": repr(e)})

all_rows = pd.concat(frames, ignore_index=True)
all_rows.to_csv(OUTPUT_DIR / "same_checkpoint_offset_epi_all.csv", index=False)
pd.DataFrame(failures).to_csv(OUTPUT_DIR / "failures.csv", index=False)
print("Rows:", len(all_rows), "| failures:", len(failures))


Electricity | fixed checkpoint H=720 | C=321 | targets=6 | sources=93 | test=256
  batch 1/32
  batch 11/32
  batch 21/32
  batch 31/32
Weather | fixed checkpoint H=720 | C=21 | targets=10 | sources=21 | test=256
  batch 1/8
Solar | fixed checkpoint H=720 | C=137 | targets=6 | sources=60 | test=256
  batch 1/16
  batch 11/16
ETTh1 | fixed checkpoint H=720 | C=7 | targets=7 | sources=7 | test=256
  batch 1/8
ETTm1 | fixed checkpoint H=720 | C=7 | targets=7 | sources=7 | test=256
  batch 1/8
Rows: 1860 | failures: 0


## 5. Cross-offset rank stability and P–EPI alignment

In [9]:
def safe_spearman(a, b):
    a = pd.Series(np.asarray(a, dtype=float)); b = pd.Series(np.asarray(b, dtype=float))
    if len(a) < 2 or a.nunique() < 2 or b.nunique() < 2: return np.nan
    return float(a.corr(b, method="spearman"))

def top_ids(df, score_col, k=RANK_K):
    return set(df.sort_values(score_col, ascending=False).head(min(k, len(df)))["source_channel"].astype(int).tolist())

def jaccard(a, b):
    return len(a & b) / max(len(a | b), 1)

pair_rows, align_rows = [], []
for (d, t), sub in all_rows.groupby(["dataset", "target_channel"]):
    # P-vs-EPI at each offset
    for h in OFFSETS:
        ss = sub[sub["offset"] == h]
        align_rows.append({
            "dataset": d, "target_channel": int(t), "offset": int(h),
            "rho_P_vs_EPI": safe_spearman(ss["predictive_utility_%"], ss["same_checkpoint_epi_%"]),
            "jaccard5_P_vs_EPI": jaccard(top_ids(ss, "predictive_utility_%"), top_ids(ss, "same_checkpoint_epi_%")),
        })
    # EPI ranking stability between offsets
    for a_i in range(len(OFFSETS)):
        for b_i in range(a_i+1, len(OFFSETS)):
            ha, hb = OFFSETS[a_i], OFFSETS[b_i]
            A = sub[sub["offset"] == ha][["source_channel", "same_checkpoint_epi_%"]].rename(columns={"same_checkpoint_epi_%":"Ia"})
            B = sub[sub["offset"] == hb][["source_channel", "same_checkpoint_epi_%"]].rename(columns={"same_checkpoint_epi_%":"Ib"})
            m = A.merge(B, on="source_channel", validate="one_to_one")
            pair_rows.append({
                "dataset": d, "target_channel": int(t), "offset_a": ha, "offset_b": hb,
                "rho_epi": safe_spearman(m["Ia"], m["Ib"]),
                "jaccard5_epi": jaccard(
                    set(m.sort_values("Ia", ascending=False).head(min(RANK_K,len(m)))["source_channel"]),
                    set(m.sort_values("Ib", ascending=False).head(min(RANK_K,len(m)))["source_channel"]),
                ),
            })

pair_df = pd.DataFrame(pair_rows)
align_df = pd.DataFrame(align_rows)
pair_df.to_csv(OUTPUT_DIR / "cross_offset_epi_stability.csv", index=False)
align_df.to_csv(OUTPUT_DIR / "offset_P_vs_EPI_alignment.csv", index=False)

display(pair_df.groupby(["offset_a","offset_b"], as_index=False).agg(
    median_rho=("rho_epi","median"), mean_jaccard5=("jaccard5_epi","mean"), n=("rho_epi","size")
).round(3))

display(align_df.groupby("offset", as_index=False).agg(
    median_rho_P_EPI=("rho_P_vs_EPI","median"), mean_jaccard5=("jaccard5_P_vs_EPI","mean"), n=("rho_P_vs_EPI","size")
).round(3))

print("\\nInterpretation guide:")
print("- Low cross-offset EPI stability: strong evidence that 'used' itself changes with future offset inside one fixed model.")
print("- High cross-offset EPI stability: equally informative; the model may retain a stable source strategy despite changing Related/Useful structure.")
print("- Keep this experiment only if the result materially sharpens the paper's story; do not force a preferred direction.")


,offset_a,offset_b,median_rho,mean_jaccard5,n
0,96,192,0.482,0.483,36
1,96,336,0.148,0.463,36
2,96,720,0.392,0.551,36
3,192,336,0.366,0.509,36
4,192,720,0.163,0.463,36
5,336,720,0.364,0.487,36


,offset,median_rho_P_EPI,mean_jaccard5,n
0,96,-0.115,0.339,36
1,192,-0.276,0.355,36
2,336,-0.121,0.401,36
3,720,-0.235,0.326,36


\nInterpretation guide:
- Low cross-offset EPI stability: strong evidence that 'used' itself changes with future offset inside one fixed model.
- High cross-offset EPI stability: equally informative; the model may retain a stable source strategy despite changing Related/Useful structure.
- Keep this experiment only if the result materially sharpens the paper's story; do not force a preferred direction.
